# Overfitting, Underfitting and Hyperparameter Tuning

This beginner-friendly Python notebook explains:

1. Underfitting
2. Overfitting
3. Good fit model
4. Model complexity
5. Cross-validation
6. Grid Search
7. Random Search
8. Final model comparison

Dataset used: **Breast Cancer dataset from sklearn**

No external dataset download is required.


## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report


## 2. Load Dataset

We use the Breast Cancer dataset from sklearn.

The task is to classify whether a tumor is malignant or benign.


In [ ]:
data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X.head()


## 3. Explore Dataset

In [ ]:
print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

print("\nTarget names:")
print(data.target_names)

print("\nFirst five feature names:")
print(X.columns[:5])


# What is Underfitting?

Underfitting happens when a model is too simple.

It cannot learn the pattern from the data.

Example:

A student studies only one chapter for a full exam.  
The student performs poorly in both practice questions and real exam questions.

In machine learning:

- Training accuracy is low
- Testing accuracy is low


## 4. Underfitting Example

We create a very simple Decision Tree using `max_depth=1`.

This model is too simple and may underfit.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

underfit_model = DecisionTreeClassifier(max_depth=1, random_state=42)
underfit_model.fit(X_train, y_train)

underfit_train_acc = accuracy_score(y_train, underfit_model.predict(X_train))
underfit_test_acc = accuracy_score(y_test, underfit_model.predict(X_test))

print("Underfitting Model")
print("Training Accuracy:", underfit_train_acc)
print("Testing Accuracy:", underfit_test_acc)


# What is Overfitting?

Overfitting happens when a model is too complex.

It memorizes the training data instead of learning general patterns.

Example:

A student memorizes only the exact answers from previous question papers.  
If the real exam has new questions, the student performs poorly.

In machine learning:

- Training accuracy is very high
- Testing accuracy is lower


## 5. Overfitting Example

We create a very deep Decision Tree.

This model may memorize the training data.


In [ ]:
overfit_model = DecisionTreeClassifier(random_state=42)
overfit_model.fit(X_train, y_train)

overfit_train_acc = accuracy_score(y_train, overfit_model.predict(X_train))
overfit_test_acc = accuracy_score(y_test, overfit_model.predict(X_test))

print("Overfitting Model")
print("Training Accuracy:", overfit_train_acc)
print("Testing Accuracy:", overfit_test_acc)


# Good Fit Model

A good model learns useful patterns without memorizing the training data.

In machine learning:

- Training accuracy is good
- Testing accuracy is also good
- Difference between training and testing accuracy is small


## 6. Good Fit Example

In [ ]:
good_model = DecisionTreeClassifier(max_depth=4, random_state=42)
good_model.fit(X_train, y_train)

good_train_acc = accuracy_score(y_train, good_model.predict(X_train))
good_test_acc = accuracy_score(y_test, good_model.predict(X_test))

print("Good Fit Model")
print("Training Accuracy:", good_train_acc)
print("Testing Accuracy:", good_test_acc)


## 7. Compare Underfitting, Overfitting and Good Fit

In [ ]:
comparison = pd.DataFrame({
    'Model Type': ['Underfitting', 'Overfitting', 'Good Fit'],
    'Training Accuracy': [underfit_train_acc, overfit_train_acc, good_train_acc],
    'Testing Accuracy': [underfit_test_acc, overfit_test_acc, good_test_acc]
})

comparison


In [ ]:
plt.figure(figsize=(8, 5))

x = np.arange(len(comparison['Model Type']))
width = 0.35

plt.bar(x - width/2, comparison['Training Accuracy'], width, label='Training Accuracy')
plt.bar(x + width/2, comparison['Testing Accuracy'], width, label='Testing Accuracy')

plt.xticks(x, comparison['Model Type'])
plt.ylabel('Accuracy')
plt.title('Underfitting vs Overfitting vs Good Fit')
plt.legend()
plt.ylim(0, 1.1)
plt.show()


# Effect of Model Complexity

For Decision Trees, `max_depth` controls complexity.

- Very small depth: may underfit
- Very large depth: may overfit
- Medium depth: may give good performance


## 8. Try Different max_depth Values

In [ ]:
depth_values = range(1, 16)

train_scores = []
test_scores = []

for depth in depth_values:
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)

    train_scores.append(accuracy_score(y_train, model.predict(X_train)))
    test_scores.append(accuracy_score(y_test, model.predict(X_test)))

depth_results = pd.DataFrame({
    'max_depth': list(depth_values),
    'Training Accuracy': train_scores,
    'Testing Accuracy': test_scores
})

depth_results


In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(depth_results['max_depth'], depth_results['Training Accuracy'], marker='o', label='Training Accuracy')
plt.plot(depth_results['max_depth'], depth_results['Testing Accuracy'], marker='o', label='Testing Accuracy')

plt.xlabel('max_depth')
plt.ylabel('Accuracy')
plt.title('Effect of max_depth on Model Performance')
plt.legend()
plt.grid(True)
plt.show()


# Hyperparameters

Hyperparameters are settings chosen before training the model.

Examples:

Decision Tree:
- max_depth
- min_samples_split
- min_samples_leaf

Random Forest:
- n_estimators
- max_depth
- min_samples_split


# Hyperparameter Tuning

Hyperparameter tuning means finding the best settings for a model.

It helps to:

- Improve accuracy
- Reduce overfitting
- Improve performance on new data


## 9. Cross Validation

Cross-validation gives a more reliable performance estimate.

In 5-fold cross-validation:

- Dataset is divided into 5 parts
- Model trains on 4 parts
- Model tests on 1 part
- Process repeats 5 times


In [ ]:
cv_model = DecisionTreeClassifier(max_depth=4, random_state=42)

cv_scores = cross_val_score(cv_model, X, y, cv=5)

print("Cross Validation Scores:", cv_scores)
print("Mean CV Accuracy:", cv_scores.mean())


## 10. Grid Search

Grid Search tries all possible combinations of given hyperparameters.


In [ ]:
param_grid = {
    'max_depth': [2, 3, 4, 5, 6, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_search = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy'
)

grid_search.fit(X_train, y_train)

print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest Cross Validation Accuracy:")
print(grid_search.best_score_)


## 11. Evaluate Best Grid Search Model

In [ ]:
best_grid_model = grid_search.best_estimator_

grid_pred = best_grid_model.predict(X_test)

print("Grid Search Test Accuracy:", accuracy_score(y_test, grid_pred))
print("\nClassification Report:")
print(classification_report(y_test, grid_pred))


## 12. Random Search

Random Search randomly tries selected hyperparameter combinations.

It is faster than Grid Search when there are many possibilities.


In [ ]:
param_random = {
    'n_estimators': [50, 100, 150, 200, 300],
    'max_depth': [2, 3, 4, 5, 6, 8, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_random,
    n_iter=10,
    cv=5,
    scoring='accuracy',
    random_state=42
)

random_search.fit(X_train, y_train)

print("Best Parameters:")
print(random_search.best_params_)

print("\nBest Cross Validation Accuracy:")
print(random_search.best_score_)


## 13. Evaluate Best Random Search Model

In [ ]:
best_random_model = random_search.best_estimator_

random_pred = best_random_model.predict(X_test)

print("Random Search Test Accuracy:", accuracy_score(y_test, random_pred))
print("\nClassification Report:")
print(classification_report(y_test, random_pred))


## 14. Final Model Comparison

In [ ]:
final_results = pd.DataFrame({
    'Model': [
        'Underfit Decision Tree',
        'Overfit Decision Tree',
        'Good Fit Decision Tree',
        'Grid Search Decision Tree',
        'Random Search Random Forest'
    ],
    'Test Accuracy': [
        underfit_test_acc,
        overfit_test_acc,
        good_test_acc,
        accuracy_score(y_test, grid_pred),
        accuracy_score(y_test, random_pred)
    ]
})

final_results


In [ ]:
plt.figure(figsize=(10, 5))

plt.bar(final_results['Model'], final_results['Test Accuracy'])
plt.xticks(rotation=30, ha='right')
plt.ylabel('Test Accuracy')
plt.title('Final Model Comparison')
plt.ylim(0, 1.1)
plt.show()


# 15. Summary

## Underfitting
- Model is too simple
- Training accuracy is low
- Testing accuracy is low

## Overfitting
- Model is too complex
- Training accuracy is very high
- Testing accuracy is lower

## Hyperparameter Tuning
- Finds the best model settings
- Improves accuracy
- Reduces overfitting
- Helps model perform better on unseen data


# Practice Questions

1. Change `max_depth` values and observe accuracy.
2. Try `min_samples_split` values 2, 5, and 10.
3. Use GridSearchCV for Random Forest.
4. Increase `n_iter` in RandomizedSearchCV.
5. Compare Decision Tree and Random Forest accuracy.
6. Write the difference between overfitting and underfitting.
7. Explain why cross-validation is useful.
